# Q3 - Transfer learnt model built on a pretrained LLM such as GPT-2 or TinyLlama using fine-tuning mechanisms such as PEFT (LoRA, QLoRA) etc

In [1]:
# Step 1: Install Necessary Libraries
!pip install transformers
!pip install peft
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 58.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

# Import Libraries

In [2]:
# Step 2: Import Libraries and Load Dataset
import pandas as pd
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

# Load Data

In [3]:
# Step 3: Load train.csv and valid.csv (use your own file paths here or upload them)
train_file_path = '/content/train.csv'
valid_file_path = '/content/valid.csv'

In [5]:
train_df = pd.read_csv(train_file_path)
valid_df = pd.read_csv(valid_file_path)

In [6]:
# Display first few rows of the datasets to inspect
train_df.head(), valid_df.head()

(                                             answers  \
 0  Kids who are bipolar, in their manic stages, v...   
 1                  Equifax, transunion and experian.   
 2  Women eat at least 1,200 calories daily and me...   
 3  Because Caffeine increases the stress hormone ...   
 4                                        Kent County   
 
                                                query  \
 0                     why do children get aggressive   
 1  which credit bureau is used the most for auto ...   
 2         what is the minimum healthy calorie intake   
 3                   why is coffee making gain weight   
 4                 what county is grand rapids, mi in   
 
                                         finalpassage  
 0  At the same time, despite claiming the review ...  
 1  Best Answer: both of those answers are wrong. ...  
 2  Safe Intakes. If you’re not supervised by a me...  
 3  Is coffee making you fat? If you are overweigh...  
 4  Located in Grand Rapids, Mic

In [7]:
# Step 4: Convert pandas DataFrame to HuggingFace dataset format
train_data = Dataset.from_pandas(train_df)
valid_data = Dataset.from_pandas(valid_df)

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


# Load Model

In [9]:
# Step 7: Load distilgpt2 Model
model_name = "distilgpt2"
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# If there is no padding token, set the EOS token as padding
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    # Combine query and passage to form input text
    input_texts = [
        (q if q is not None else "") + " " + (p if p is not None else "")
        for q, p in zip(examples["query"], examples["finalpassage"])
    ]

    tokenized_output = tokenizer(input_texts, padding="max_length", truncation=True, max_length=512)

    # Causal LM expects labels to be the same as input_ids
    tokenized_output["labels"] = tokenized_output["input_ids"].copy()

    return tokenized_output

# Apply tokenization on both datasets
train_data = train_data.map(tokenize_function, batched=True, batch_size=8, keep_in_memory=True)
valid_data = valid_data.map(tokenize_function, batched=True, batch_size=8, keep_in_memory=True)

# Step 8: Load Model and Move to Device
from transformers import AutoModelForCausalLM

# Load the model
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set device to GPU if available, else fallback to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# Move the model to the appropriate device
model.to(device)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10585 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [10]:
# Step 6: Sampling the data (taking random samples for testing)
train_sample = train_data.shuffle(seed=42).select([i for i in list(range(5))])  # First 5 samples
valid_sample = valid_data.shuffle(seed=42).select([i for i in list(range(5))])  # First 5 samples

# Inspect sample
train_sample, valid_sample

(Dataset({
     features: ['answers', 'query', 'finalpassage', 'input_ids', 'attention_mask', 'labels'],
     num_rows: 5
 }),
 Dataset({
     features: ['answers', 'query', 'finalpassage', 'input_ids', 'attention_mask', 'labels'],
     num_rows: 5
 }))

# Configure LoRA and Apply to the Model

In [11]:
from transformers import AutoModelForCausalLM
import torch

# Step 7: Load distilgpt2 Model
model_name = "distilgpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)

# Step 8: Configure LoRA and Apply to the Model
lora_config = LoraConfig(
    r=8,  # Rank of low-rank matrices
    lora_alpha=32,  # Scaling factor
    lora_dropout=0.05,  # Dropout rate for LoRA layers
    bias="none",  # No bias in LoRA
    task_type="CAUSAL_LM"  # Causal language modeling task
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Check if a GPU is available, otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# Move the model to the selected device (either GPU or CPU)
model.to(device)


/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/layer.py:1264: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-5): 6 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=2304, nx=768)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
            

# Generate Text using model

In [12]:
# Step 9: Generate Text Using the LoRA Model
def generate_text(prompt, max_length=50):
    # Tokenize input prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Generate text
    outputs = model.generate(
        inputs['input_ids'],
        max_length=max_length,
        num_return_sequences=1,
        no_repeat_ngram_size=2,  # Prevent repeating n-grams
        temperature=0.7,  # Sampling temperature
        top_k=50,  # Top-k sampling
        top_p=0.95,  # Top-p sampling
    )

    # Decode and return generated text
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

# Step 10: Test Generation on a Sample
sample_prompt = "Once upon a time in a land far away"
generated_sample = generate_text(sample_prompt)
print(generated_sample)


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpe

Once upon a time in a land far away, the sun is shining.

The sun rises and the moon rises. The sun sets. It is the night of the day.


In [13]:
# Example of selecting a random 10% sample of the dataset
train_data_sampled = train_data.shuffle(seed=42).select([i for i in range(int(len(train_data) * 0.1))])
valid_data_sampled = valid_data.shuffle(seed=42).select([i for i in range(int(len(valid_data) * 0.1))])


# Fine Tuning and train the model

In [14]:
 # Step 11: Fine-tune the model (Optional) - Fine-tuning on the dataset can be done here

from transformers import Trainer, TrainingArguments

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',         # output directory
    num_train_epochs=3,             # number of training epochs
    per_device_train_batch_size=8,  # batch size for training
    per_device_eval_batch_size=8,   # batch size for evaluation
    logging_dir='./logs',           # directory for storing logs
)

# Use sampled datasets for training
trainer = Trainer(
    model=model,                         # the model to train
    args=training_args,                  # training arguments
    train_dataset=train_data_sampled,     # sampled training dataset
    eval_dataset=valid_data_sampled,      # sampled validation dataset
)

# Train the model
trainer.train()



No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jissyj1996 (jissyj1996-loyalist-college) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,2.720200
1000,1.244800
1500,1.224100
2000,1.213800
2500,1.221900
3000,1.219500
3500,1.208600
4000,1.213000
4500,1.214600


TrainOutput(global_step=4500, training_loss=1.3867266167534722, metrics={'train_runtime': 2574.7034, 'train_samples_per_second': 13.982, 'train_steps_per_second': 1.748, 'total_flos': 4719648964608000.0, 'train_loss': 1.3867266167534722, 'epoch': 3.0})

In [16]:
!pip install evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.4 MB/s eta 0:00:00


In [20]:
!pip install rouge_score


  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=75f278d0fbfaab56220cfe545b48e2f161ead4c4b00b508392698d5dbed0678c
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [22]:
!pip install bert_score


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.6 MB/s eta 0:00:00


# Evaluation using ROUGE and BERTScore metrics

In [23]:
import evaluate

# Load ROUGE and BERTScore metrics
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

# Function to evaluate the model performance using ROUGE-L and BERT-F1
def evaluate_model(model, tokenizer, dataset):
    predictions = []
    references = []

    for example in dataset:
        input_text = example["query"]
        target_text = example["finalpassage"]

        # Generate response from the model
        inputs = tokenizer(input_text, return_tensors="pt").to(device)
        generated_ids = model.generate(inputs['input_ids'], max_length=150)
        generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

        predictions.append(generated_text)
        references.append(target_text)

    # Compute ROUGE-L and BERT-F1
    rouge_result = rouge.compute(predictions=predictions, references=references)
    bertscore_result = bertscore.compute(predictions=predictions, references=references, model_type="bert-base-uncased")

    return rouge_result, bertscore_result

# Evaluate the fine-tuned model on the validation set
rouge_result, bertscore_result = evaluate_model(model, tokenizer, valid_sample)

# Output evaluation results
print(f"ROUGE-L: {rouge_result}")
print(f"BERTScore F1: {bertscore_result['f1']}")


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generati

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

ROUGE-L: {'rouge1': np.float64(0.14647387141305052), 'rouge2': np.float64(0.024852071005917162), 'rougeL': np.float64(0.12429599903963232), 'rougeLsum': np.float64(0.12429599903963232)}
BERTScore F1: [0.4410865604877472, 0.48622220754623413, 0.3779492974281311, 0.5065504908561707, 0.5807270407676697]


In [29]:
model.save_pretrained("/content/fine_tuned_model")
tokenizer.save_pretrained("/content/fine_tuned_model")


('/content/fine_tuned_model/tokenizer_config.json',
 '/content/fine_tuned_model/special_tokens_map.json',
 '/content/fine_tuned_model/vocab.json',
 '/content/fine_tuned_model/merges.txt',
 '/content/fine_tuned_model/added_tokens.json',
 '/content/fine_tuned_model/tokenizer.json')

In [30]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [31]:
model.save_pretrained("/content/drive/MyDrive/fine_tuned_model")
tokenizer.save_pretrained("/content/drive/MyDrive/fine_tuned_model")


('/content/drive/MyDrive/fine_tuned_model/tokenizer_config.json',
 '/content/drive/MyDrive/fine_tuned_model/special_tokens_map.json',
 '/content/drive/MyDrive/fine_tuned_model/vocab.json',
 '/content/drive/MyDrive/fine_tuned_model/merges.txt',
 '/content/drive/MyDrive/fine_tuned_model/added_tokens.json',
 '/content/drive/MyDrive/fine_tuned_model/tokenizer.json')